In [ ]:

#!/usr/bin/env python
"""
回测系统入口脚本。
"""

# from __future__ import annotations

# import argparse
import json
from pathlib import Path
from typing import Any, Dict

import yaml

from Backtest.backtest_engine import run_backtest
from Backtest.prediction_metrics import evaluate_prediction_performance
from Backtest.signal_generator import generate_signals, save_signals
from Backtest.visualization import (
    plot_backtest_figures,
    plot_dashboard_table,
    plot_prediction_figures,
)

import numpy as np
import polars as pl
import torch

def load_yaml(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    return cfg


backet_config="Configs/backtest_config.yaml"
backtest_config = load_yaml(backet_config)

In [2]:
output_dir = Path(backtest_config.get("output_dir", "backtest_results/"))
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
backtest_config

{'model_checkpoint': 'checkpoints/multi_modal_model_2/seed_42/best_model.pt',
 'model_config': '/root/lio/Trade_LOB_MultiModal/checkpoints/multi_modal_model_2/model_config.yaml',
 'data': {'lob_path': '/root/autodl-tmp/test_data/lob_data.npy',
  'trade_path': '/root/autodl-tmp/test_data/trade_data.npy',
  'label_path': '/root/autodl-tmp/test_data/trade_labels_ret.npy'},
 'signal_config': {'history_T': 3000,
  'signal_stride': 100,
  'batch_size': 512,
  'num_workers': 4,
  'pin_memory': True,
  'alpha': 0.001,
  'use_amp': True},
 'prediction_eval': {'ic_rolling_window': 500, 'confidence_bins': 5},
 'backtest': {'strategy': 'discrete',
  'fee_rate': 0.0001,
  'slippage': 0.0,
  'score_threshold': 0.0,
  'risk_free_rate': 0.0},
 'output_dir': 'backtest_results/'}

In [12]:
signal_cfg = backtest_config.get("signal_config", {})
data_cfg = backtest_config.get("data", {})

history_t = int(signal_cfg.get("history_T", 3000))
signal_stride = int(signal_cfg.get("signal_stride", 100))
batch_size = int(signal_cfg.get("batch_size", 512))
num_workers = int(signal_cfg.get("num_workers", 4))
pin_memory = bool(signal_cfg.get("pin_memory", True))
alpha = float(signal_cfg.get("alpha", 0.001))

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = bool(signal_cfg.get("use_amp", True)) and device.startswith("cuda")

lob_data = np.load(data_cfg["lob_path"])
trade_path = data_cfg.get("trade_path")
trade_data = np.load(trade_path) if trade_path else None
labels_ret = np.load(data_cfg["label_path"])

In [14]:
backet_config

'Configs/backtest_config.yaml'

In [15]:
model_config_path = backtest_config['model_config']
checkpoint_path = backtest_config['model_checkpoint']

In [17]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

In [32]:
if "model_state_dict" not in checkpoint:
    raise KeyError(f"checkpoint 缺少 model_state_dict: {checkpoint_path}")
state_dict = checkpoint["model_state_dict"]

with open(model_config_path, "r", encoding="utf-8") as f:
    model_cfg = yaml.safe_load(f)

use_trade = model_cfg['trade_encoder'] is not None

In [28]:
from Model import MultiModalTransformer

model = MultiModalTransformer.from_config(model_config_path)

model.load_state_dict(state_dict, strict=True)
model.to(device)
model.eval()

MultiModalTransformer(
  (lob_encoder): LOBEncoder(
    (blocks): ModuleList(
      (0): CausalDownsamplingBlock(
        (spatial_conv): Sequential(
          (0): Conv2d(4, 32, kernel_size=(1, 2), stride=(1, 2))
          (1): GroupNorm(2, 32, eps=1e-05, affine=True)
          (2): LeakyReLU(negative_slope=0.01)
          (3): Dropout2d(p=0.1, inplace=False)
        )
        (time_conv): CausalConv2d(
          (conv): Conv2d(32, 32, kernel_size=(3, 1), stride=(5, 1))
        )
        (norm_time): GroupNorm(2, 32, eps=1e-05, affine=True)
        (act): LeakyReLU(negative_slope=0.01)
        (dropout): Dropout2d(p=0.1, inplace=False)
      )
      (1): CausalDownsamplingBlock(
        (spatial_conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(1, 2), stride=(1, 2))
          (1): GroupNorm(2, 32, eps=1e-05, affine=True)
          (2): LeakyReLU(negative_slope=0.01)
          (3): Dropout2d(p=0.1, inplace=False)
        )
        (time_conv): CausalConv2d(
          (conv)

In [29]:
# 打印模型信息
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数量: {total_params:,} (可训练: {trainable_params:,})")

模型参数量: 28,011 (可训练: 28,011)


In [36]:
test_dict = {"lob": lob_data}
if use_trade and trade_data is not None:
    test_dict["trade"] = trade_data

test_labels_ret = labels_ret
test_labels_class = np.ones_like(test_labels_ret, dtype=np.int64)
test_labels_class[labels_ret > alpha] = 2
test_labels_class[labels_ret < -alpha] = 0

In [37]:
loader_cfg = {
        "history_T": history_t,
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": pin_memory,
        "stride": signal_stride,
    }


from Data_Pipeline.dataset import create_dataloaders_for_test
test_loader = create_dataloaders_for_test(
    data_dict=test_dict,
    labels=test_labels_class,
    returns=test_labels_ret,
    config=loader_cfg,
    device="cpu",
)

In [38]:
test_mid_price = (test_dict["lob"][:, 0, 0] + test_dict["lob"][:, 1, 0]) / 2.0

pred_classes = []
pred_probas = []
actual_returns = []
tick_indices = []
mid_prices = []

In [39]:
from torch.amp import autocast
from tqdm import tqdm
with torch.no_grad():
    global_row = 0
    for inputs, _, returns in tqdm(test_loader, desc="Generating signals", leave=False):
        if not isinstance(inputs, dict):
            raise TypeError("输入格式异常，期望 dict[str, Tensor]。")
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

        with autocast(device_type="cuda", enabled=use_amp):
            logits = model(inputs)
            probas = torch.softmax(logits, dim=1)

        preds = torch.argmax(probas, dim=1).cpu().numpy()
        probas_np = probas.cpu().numpy()
        returns_np = returns.numpy()

        batch_now = preds.shape[0]
        for j in range(batch_now):
            dataset_idx = global_row + j
            start_idx_in_test = dataset_idx * signal_stride
            test_tick_idx = (history_t - 1) + start_idx_in_test
            if test_tick_idx >= len(test_mid_price):
                continue

            raw_tick_idx = test_tick_idx
            tick_indices.append(int(raw_tick_idx))
            pred_classes.append(int(preds[j]))
            pred_probas.append(probas_np[j])
            actual_returns.append(float(returns_np[j]))
            mid_prices.append(float(test_mid_price[test_tick_idx]))

        global_row += batch_now
        break

In [49]:
preds.shape

(512,)